# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisal-0065/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will rank content based on observed search performance. Content with lower CTR and lower impressions will receive a higher priority for review.

The two signals I checked are CTR and impressions. The results show that many rows have zero impressions, while most rows with measurable CTR fall into the low CTR bucket.

Verdicts:

CTR — CONFIRMED
Impressions — CONFIRMED

The rule will use one reason code: PERFORMANCE_REVIEW.

This is a directional decision-support score based on observed data. It does not prove causation or predict future Google performance.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [2]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print("Shape:", march_df.shape)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Dataset loaded successfully!
Shape: (9841378, 30)


In [3]:
print("Available columns:")
print(march_df.columns.tolist())

print("\nGSC impressions summary:")
print(march_df["gsc_impressions"].describe())

print("\nGSC clicks summary:")
print(march_df["gsc_clicks"].describe())

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

GSC impressions summary:
count    9.841378e+06
mean     2.851812e+01
std      1.559266e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.000000e+00
max      4.008400e+04
Name: gsc_impressions, dtype: float64

GSC clicks summary:
count    9.841378e+06
mean     8.350782e-02
std      7.814341e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.740000e+02
Name: gsc_clicks, 

In [4]:
import numpy as np

# Calculate CTR
march_df["ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
)

# Create CTR buckets
march_df["ctr_bucket"] = pd.cut(
    march_df["ctr"],
    bins=[-np.inf, 0.01, 0.05, np.inf],
    labels=["low", "medium", "high"]
)

# Create impression buckets
march_df["impression_bucket"] = pd.cut(
    march_df["gsc_impressions"],
    bins=[-np.inf, 0, 10, np.inf],
    labels=["zero", "low", "high"]
)

print("CTR bucket table:")
print(march_df["ctr_bucket"].value_counts(dropna=False))

print("\nImpression bucket table:")
print(march_df["impression_bucket"].value_counts(dropna=False))

CTR bucket table:
ctr_bucket
NaN       6230317
low       3414297
medium     162052
high        34712
Name: count, dtype: int64

Impression bucket table:
impression_bucket
zero    6230317
high    2079427
low     1531634
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# Build the baseline action score

# Calculate CTR
march_df["ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
)

# Normalize the two signals
# Lower CTR = more attention needed
ctr_score = 1 - march_df["ctr"].clip(upper=1).fillna(0)

# Lower impressions = more attention needed
impression_score = 1 / (1 + march_df["gsc_impressions"])

# Combine the signals into one score
march_df["baseline_score"] = (
    0.6 * ctr_score +
    0.4 * impression_score
)

# One reason code
march_df["reason_code"] = "PERFORMANCE_REVIEW"

# Action label
march_df["action"] = "REVIEW_CONTENT"

# Rank from highest score to lowest
queue = march_df[
    [
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "baseline_score",
        "reason_code",
        "action"
    ]
].sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# Add rank
queue.insert(0, "rank", range(1, len(queue) + 1))

# Show top 10
print("Top 10 baseline recommendations:")
display(queue.head(10))


Top 10 baseline recommendations:


,rank,content_hash_id,client_hash_id,report_date,baseline_score,reason_code,action
0,1,content_f39be42b42a4e8f6,client_73cda7b4e4f265ea,2026-03-01,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
1,2,content_d105e79bf855829c,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
2,3,content_30b9b58a0675f620,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
3,4,content_0ea64f25303c9a77,client_73cda7b4e4f265ea,2026-03-01,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
4,5,content_c480a5f3a909175e,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
5,6,content_dd1deec61eb35197,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
6,7,content_0b88e713bfa1eae9,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
7,8,content_a5b7be083fcba9a4,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
8,9,content_31898aedf6c9662b,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
9,10,content_dfc972c4709a0b42,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT


In [6]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Output folder ready!")

Output folder ready!


In [7]:
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Saved successfully!")
print("File:", output_path)
print("Rows:", len(queue))

Saved successfully!
File: work/outputs/baseline_action_score.csv
Rows: 9841378


In [8]:
# Show the top 20 recommendations

top20 = queue.head(20).copy()

display(top20)

,rank,content_hash_id,client_hash_id,report_date,baseline_score,reason_code,action
0,1,content_f39be42b42a4e8f6,client_73cda7b4e4f265ea,2026-03-01,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
1,2,content_d105e79bf855829c,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
2,3,content_30b9b58a0675f620,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
3,4,content_0ea64f25303c9a77,client_73cda7b4e4f265ea,2026-03-01,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
4,5,content_c480a5f3a909175e,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
5,6,content_dd1deec61eb35197,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
6,7,content_0b88e713bfa1eae9,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
7,8,content_a5b7be083fcba9a4,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
8,9,content_31898aedf6c9662b,client_20259bd6705d81d4,2026-03-21,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT
9,10,content_dfc972c4709a0b42,client_20259bd6705d81d4,2026-03-31,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



I reviewed the top 20 rows from the baseline queue. Each row is marked for content review because of the observed performance signals used by the baseline score.

The recommendations are directional and should be checked by a human before taking action. A recommendation could be wrong if the content has a different business purpose, if the observed search data is incomplete, or if the low performance is explained by another factor such as search position.


In [9]:
# Create a simple review table for the top 20

top20_review = top20[
    ["rank", "content_hash_id", "baseline_score", "reason_code", "action"]
].copy()

top20_review["confidence_note"] = (
    "Moderate confidence based on observed March performance signals."
)

top20_review["what_would_make_it_wrong"] = (
    "Incomplete data, unusual content purpose, or another factor affecting performance."
)

display(top20_review)


,rank,content_hash_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_f39be42b42a4e8f6,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
1,2,content_d105e79bf855829c,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
2,3,content_30b9b58a0675f620,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
3,4,content_0ea64f25303c9a77,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
4,5,content_c480a5f3a909175e,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
5,6,content_dd1deec61eb35197,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
6,7,content_0b88e713bfa1eae9,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
7,8,content_a5b7be083fcba9a4,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
8,9,content_31898aedf6c9662b,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."
9,10,content_dfc972c4709a0b42,1.0,PERFORMANCE_REVIEW,REVIEW_CONTENT,Moderate confidence based on observed March pe...,"Incomplete data, unusual content purpose, or a..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



Some top picks may be weak because the score is based on only two observed performance signals. A low score signal does not always mean the content needs to be changed.

The baseline does not use future performance, trend labels, or target-derived fields. The score uses March 2026 observed search performance only.

A limitation is that the rule does not know the content's full business purpose, so the ranked recommendations should be reviewed by a person before action.


In [10]:
# Check for weak picks and possible leakage

print("Top 20 score range:")
print(top20["baseline_score"].min(), "to", top20["baseline_score"].max())

print("\nReason codes in top 20:")
print(top20["reason_code"].value_counts())

print("\nAction labels in top 20:")
print(top20["action"].value_counts())

# Check that no future-window or label-derived columns were used
leakage_columns = [
    col for col in march_df.columns
    if any(word in col.lower() for word in [
        "future", "trend", "label", "target"
    ])
]

print("\nPotential leakage columns found:")
print(leakage_columns)


Top 20 score range:
1.0 to 1.0

Reason codes in top 20:
reason_code
PERFORMANCE_REVIEW    20
Name: count, dtype: int64

Action labels in top 20:
action
REVIEW_CONTENT    20
Name: count, dtype: int64

Potential leakage columns found:
[]
